# Camada Silver — Limpeza, Padronização e Enriquecimento
**Regras gerais:**
- Nenhuma tabela Bronze é alterada.
- Todos os nomes de colunas em português.
- Tipagem correta em todas as colunas.
- Deduplicação via Window Function (registro mais recente por timestamp_ingestion).

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.types import DoubleType, IntegerType, DateType


In [0]:
catalogo = 'olist_catalog'
bronze_db_name = 'bronze'
silver_db_name = 'silver'

In [0]:
spark.sql(f'USE CATALOG {catalogo}')

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {silver_db_name}')
spark.sql(f'USE SCHEMA {silver_db_name}')

In [0]:
# Retém apenas o registro mais recente por chave primária, usando row_number() com ordenação
# decrescente por timestamp_ingestion. Necessário porque a ingestão Bronze opera em modo append.
def deduplicate(df, partition_col: str):
    w = Window.partitionBy(partition_col).orderBy(F.col('timestamp_ingestion').desc())
    df_dedup = (
        df.withColumn('_rn', F.row_number().over(w))
          .filter(F.col('_rn') == 1)
          .drop('_rn', 'timestamp_ingestion')
    )
    qtd_removidos = df.count() - df_dedup.count()
    print(f'Valores removidos na deduplicação ({partition_col}): {qtd_removidos}')
    return df_dedup

In [0]:
# silver.dim_consumidores — origem: bronze.tb_customers
df_customers = spark.table(f'{bronze_db_name}.tb_customers')

df_dim_consumidores = df_customers.select(
    F.col('customer_id').alias('id_consumidor'),
    F.col('customer_zip_code_prefix').cast(IntegerType()).alias('prefixo_cep'),
    F.col('customer_unique_id').alias('id_consumidor_unico'),
    F.initcap(F.col('customer_name')).alias('nome_consumidor'),
    F.upper(F.col('customer_city')).alias('cidade'),
    F.upper(F.col('customer_state')).alias('estado'),
    'timestamp_ingestion'
)

df_dim_consumidores = deduplicate(df_dim_consumidores, 'id_consumidor')

(
    df_dim_consumidores.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.dim_consumidores')
)
print(f'silver.dim_consumidores: {df_dim_consumidores.count()} registros.')

In [0]:
# silver.fat_pedidos — origem: bronze.tb_orders
# O status é traduzido para português via expressão CASE construída dinamicamente a partir do dicionário.
# As colunas derivadas tempo_entrega_dias, tempo_entrega_estimado_dias e diferenca_entrega_dias
# permitem calcular entrega_no_prazo sem joins adicionais nas camadas superiores.
df_orders = spark.table(f'{bronze_db_name}.tb_orders')

# Mapa de tradução de status
status_map = {
    'delivered':    'entregue',
    'canceled':     'cancelado',
    'shipped':      'enviado',
    'processing':   'processando',
    'invoiced':     'faturado',
    'unavailable':  'indisponível',
    'created':      'criado',
    'approved':     'aprovado',
}

# Cria expressão CASE dinâmica a partir do dicionário
status_expr = F.col('order_status')
for en, pt in status_map.items():
    status_expr = F.when(F.col('order_status') == en, pt).otherwise(status_expr)

df_fat_pedidos = df_orders.select(
    F.col('order_id').alias('id_pedido'),
    F.col('customer_id').alias('id_consumidor'),
    status_expr.alias('status'),
    F.to_timestamp('order_purchase_timestamp').alias('data_pedido'),
    F.to_timestamp('order_approved_at').alias('data_aprovacao'),
    F.to_timestamp('order_delivered_carrier_date').alias('data_entrega_transportadora'),
    F.to_timestamp('order_delivered_customer_date').alias('data_entrega_cliente'),
    F.to_timestamp('order_estimated_delivery_date').alias('data_estimada_entrega'),
).withColumn(
    # Dias entre entrega real e compra
    'tempo_entrega_dias',
    F.datediff('data_entrega_cliente', 'data_pedido').cast(IntegerType())
).withColumn(
    # Dias entre estimativa e compra
    'tempo_entrega_estimado_dias',
    F.datediff('data_estimada_entrega', 'data_pedido').cast(IntegerType())
).withColumn(
    # Diferença entre real e estimado (negativo = entregue antes do prazo)
    'diferenca_entrega_dias',
    F.col('tempo_entrega_dias') - F.col('tempo_entrega_estimado_dias')
).withColumn(
    # Entregue no prazo?
    'entrega_no_prazo',
    F.when(F.col('status') != 'entregue', 'Não Entregue')
     .when(F.col('diferenca_entrega_dias') <= 0, 'Sim')
     .otherwise('Não')
)

(
    df_fat_pedidos.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.fat_pedidos')
)
print(f'{silver_db_name}.fat_pedidos: {df_fat_pedidos.count()} registros.')

In [0]:
# silver.fat_itens_pedidos — origem: bronze.tb_order_items
df_items = spark.table(f'{bronze_db_name}.tb_order_items')

df_fat_itens = df_items.select(
    F.col('order_id').alias('id_pedido'),
    F.col('order_item_id').cast(IntegerType()).alias('id_item'),
    F.col('product_id').alias('id_produto'),
    F.col('seller_id').alias('id_vendedor'),
    F.to_timestamp('shipping_limit_date').alias('data_limite_envio'),
    F.col('price').cast(DoubleType()).alias('preco_BRL'),
    F.col('freight_value').cast(DoubleType()).alias('preco_frete'),
)

(
    df_fat_itens.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.fat_itens_pedidos')
)
print(f'{silver_db_name}.fat_itens_pedidos: {df_fat_itens.count()} registros.')

In [0]:
# silver.fat_pagamentos_pedidos — origem: bronze.tb_order_payments
# O tipo de pagamento é traduzido para português via expressão CASE dinâmica.
df_payments = spark.table(f'{bronze_db_name}.tb_order_payments')

payment_map = {
    'credit_card': 'Cartão de Crédito',
    'boleto':      'Boleto',
    'voucher':     'Voucher',
    'debit_card':  'Cartão de Débito',
    'not_defined': 'Não Definido',
}

payment_expr = F.col('payment_type')
for en, pt in payment_map.items():
    payment_expr = F.when(F.col('payment_type') == en, pt).otherwise(payment_expr)

df_fat_pagamentos = df_payments.select(
    F.col('order_id').alias('id_pedido'),
    F.col('payment_sequential').cast(IntegerType()).alias('sequencia_pagamento'),
    payment_expr.alias('tipo_pagamento'),
    F.col('payment_installments').cast(IntegerType()).alias('parcelas'),
    F.col('payment_value').cast(DoubleType()).alias('valor_pagamento'),
)

(
    df_fat_pagamentos.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.fat_pagamentos_pedidos')
)
print(f'{silver_db_name}.fat_pagamentos_pedidos: {df_fat_pagamentos.count()} registros.')

In [0]:
# silver.fat_avaliacoes_pedidos — origem: bronze.tb_order_reviews
# try_to_timestamp é usado no lugar de to_timestamp para tolerar datas malformadas sem falhar o job.
# Avaliações com data_criacao no futuro são removidas por serem inconsistentes.
df_reviews = spark.table(f'{bronze_db_name}.tb_order_reviews')

df_fat_avaliacoes = (
    df_reviews
    .select(
        F.col('review_id').alias('id_avaliacao'),
        F.col('order_id').alias('id_pedido'),
        F.col('review_score').cast(IntegerType()).alias('nota_avaliacao'),
        F.col('review_comment_title').alias('_titulo_raw'),
        F.col('review_comment_message').alias('_msg_raw'),
        F.try_to_timestamp('review_creation_date').alias('data_criacao_avaliacao'),
        F.try_to_timestamp('review_answer_timestamp').alias('data_resposta_avaliacao'),
    )
    # Tratamento dos campos de texto: usa as colunas raw
    .withColumn(
        'titulo_avaliacao',
        F.when(
            F.col('_titulo_raw').isNull() | (F.trim(F.col('_titulo_raw')) == ''),
            F.lit('Sem título')
        ).otherwise(F.col('_titulo_raw'))
    )
    .withColumn(
        'comentario_avaliacao',
        F.when(
            F.col('_msg_raw').isNull() | (F.trim(F.col('_msg_raw')) == ''),
            F.lit('Sem comentário')
        ).otherwise(F.col('_msg_raw'))
    )
    .drop('_titulo_raw', '_msg_raw')
    # Remove pedidos com id nulo (inválido)
    .filter(F.col('id_pedido').isNotNull())
    # Remove avaliações com data de criação no futuro
    .filter(
        F.col('data_criacao_avaliacao').isNull() |
        (F.col('data_criacao_avaliacao') <= F.current_timestamp())
    )
    .select(
        'id_avaliacao', 'id_pedido', 'nota_avaliacao',
        'titulo_avaliacao', 'comentario_avaliacao',
        'data_criacao_avaliacao', 'data_resposta_avaliacao'
    )
)

(
    df_fat_avaliacoes.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.fat_avaliacoes_pedidos')
)
print(f'{silver_db_name}.fat_avaliacoes_pedidos: {df_fat_avaliacoes.count()} registros.')

In [0]:
# silver.dim_produtos — origem: bronze.tb_products
df_products = spark.table(f'{bronze_db_name}.tb_products')

df_dim_produtos = df_products.select(
    F.col('product_id').alias('id_produto'),
    F.col('product_category_name').alias('categoria_produto'),
    F.col('product_name_lenght').cast(IntegerType()).alias('tamanho_nome_produto'),
    F.col('product_description_lenght').cast(IntegerType()).alias('tamanho_descricao_produto'),
    F.col('product_photos_qty').cast(IntegerType()).alias('quantidade_fotos'),
    F.col('product_weight_g').cast(IntegerType()).alias('peso_produto_gramas'),
    F.col('product_length_cm').cast(IntegerType()).alias('comprimento_centimetros'),
    F.col('product_height_cm').cast(IntegerType()).alias('altura_centimetros'),
    F.col('product_width_cm').cast(IntegerType()).alias('largura_centimetros'),
    'timestamp_ingestion'
)

df_dim_produtos = deduplicate(df_dim_produtos, 'id_produto')

(
    df_dim_produtos.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.dim_produtos')
)
print(f'{silver_db_name}.dim_produtos: {df_dim_produtos.count()} registros.')

In [0]:
# silver.dim_vendedores — origem: bronze.tb_sellers
df_sellers = spark.table(f'{bronze_db_name}.tb_sellers')

df_dim_vendedores = df_sellers.select(
    F.col('seller_id').alias('id_vendedor'),
    F.col('seller_zip_code_prefix').cast(IntegerType()).alias('prefixo_cep'),
    F.upper(F.col('seller_city')).alias('cidade'),
    F.upper(F.col('seller_state')).alias('estado'),
    'timestamp_ingestion'
)

df_dim_vendedores = deduplicate(df_dim_vendedores, 'id_vendedor')

(
    df_dim_vendedores.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.dim_vendedores')
)
print(f'{silver_db_name}.dim_vendedores: {df_dim_vendedores.count()} registros.')

In [0]:
# silver.dim_categoria_produtos_traducao — origem: bronze.tb_product_category_name_translation
df_cat_trad = spark.table(f'{bronze_db_name}.tb_product_category_name_translation')

df_dim_cat = df_cat_trad.select(
    F.col('product_category_name').alias('nome_produto_pt'),
    F.col('product_category_name_english').alias('nome_produto_en'),
)

(
    df_dim_cat.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.dim_categoria_produtos_traducao')
)
print(f'{silver_db_name}.dim_categoria_produtos_traducao: {df_dim_cat.count()} registros.')

In [0]:
# silver.dim_cotacao_dolar — origem: bronze.tb_cotacao_dolar
# A API PTAX só retorna dias úteis, deixando lacunas em fins de semana.
# Para evitar NULLs ao fazer join com datas de pedido, é gerado um calendário contínuo
# e aplicado forward-fill: sábado e domingo herdam a última cotação válida (sexta-feira).
df_cotacao_raw = (
    spark.table(f'{bronze_db_name}.tb_cotacao_dolar')
         .withColumn('data_cotacao', F.to_date('data_hora_cotacao'))
         .groupBy('data_cotacao')
         .agg(F.last('cotacao_compra').alias('cotacao_compra'))  # última cotação do dia
)

# Determina intervalo de datas presentes na cotação
min_max = df_cotacao_raw.agg(
    F.min('data_cotacao').alias('min_dt'),
    F.max('data_cotacao').alias('max_dt')
).collect()[0]

# Gera calendário contínuo (todos os dias, inclusive fins de semana)
df_calendario = spark.sql(
    f"""SELECT explode(sequence(
            to_date('{min_max.min_dt}'),
            to_date('{min_max.max_dt}'),
            interval 1 day
        )) AS data_cotacao"""
)

# Join left: dias sem cotação ficam com NULL
df_cotacao_full = df_calendario.join(df_cotacao_raw, 'data_cotacao', 'left')

# Forward-fill: usa a última cotação válida (sexta) para sábado e domingo
w_ff = (
    Window.orderBy('data_cotacao')
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_dim_cotacao = (
    df_cotacao_full
    .withColumn(
        'cotacao_compra',
        F.last('cotacao_compra', ignorenulls=True).over(w_ff)
    )
    .withColumn('cotacao_compra', F.round('cotacao_compra', 4))
    .orderBy('data_cotacao')
)

(
    df_dim_cotacao.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.dim_cotacao_dolar')
)
print(f'{silver_db_name}.dim_cotacao_dolar: {df_dim_cotacao.count()} datas geradas.')

In [0]:
# silver.fat_pedido_total — consolida fat_pedidos, fat_pagamentos_pedidos e dim_cotacao_dolar.
# O join com a cotação é feito pela data do pedido (cast para DateType), aproveitando o
# calendário contínuo criado acima para garantir que não haverá NULLs de fim de semana.
# Valores financeiros arredondados para 2 casas decimais.
df_pedidos    = spark.table(f'{silver_db_name}.fat_pedidos')
df_pagamentos = spark.table(f'{silver_db_name}.fat_pagamentos_pedidos')
df_cotacao    = spark.table(f'{silver_db_name}.dim_cotacao_dolar')

# Agrega pagamentos por pedido
df_pag_agg = (
    df_pagamentos
    .groupBy('id_pedido')
    .agg(F.sum('valor_pagamento').alias('valor_total_pago_brl'))
)

df_fat_pedido_total = (
    df_pedidos
    .join(df_pag_agg, 'id_pedido', 'left')
    .join(
        df_cotacao,
        df_pedidos.data_pedido.cast(DateType()) == df_cotacao.data_cotacao,
        'left'
    )
    .withColumn(
        'valor_total_pago_brl',
        F.round('valor_total_pago_brl', 2)              # 2 casas decimais obrigatório
    )
    .withColumn(
        'valor_total_pago_usd',
        F.round(
            F.col('valor_total_pago_brl') / F.col('cotacao_compra'), 2
        )                                               # conversão para USD
    )
    .select(
        'id_pedido', 'id_consumidor', 'status',
        'valor_total_pago_brl', 'valor_total_pago_usd',
        'data_pedido'
    )
)

(
    df_fat_pedido_total.write
        .format('delta').mode('overwrite')
        .option('overwriteSchema', 'true')
        .saveAsTable(f'{silver_db_name}.fat_pedido_total')
)
print(f'{silver_db_name}.fat_pedido_total: {df_fat_pedido_total.count()} registros.')

In [0]:
# OPTIMIZE + ZORDER nas tabelas fato principais.
# ZORDER por id_pedido e data_pedido reduz o data skipping em consultas analíticas típicas.
tabelas_para_otimizar = [
    (f'{silver_db_name}.fat_pedidos',       'id_pedido, data_pedido'),
    (f'{silver_db_name}.fat_pedido_total',  'id_pedido, data_pedido'),
    (f'{silver_db_name}.fat_itens_pedidos', 'id_pedido'),
]

for tabela, colunas in tabelas_para_otimizar:
    print(f'Otimizando {tabela} (ZORDER BY {colunas})...')
    spark.sql(f'OPTIMIZE {tabela} ZORDER BY ({colunas})')
    print(f'    {tabela} otimizada.')